# Five-class LLM authorship assignment

**Experiment 1:** four classifiers on original code. **Experiment 2:** repeat without comments. **Experiment 3 (optional):** CodeT5 on original code.

Experiments 1 and 2 need only CPU. Experiment 3 needs a GPU and can take many hours. Choose the GPU runtime before running setup if you plan to run all three. To run only CodeT5, run Setup, Shared code and Data preparation, then jump to Experiment 3.

The first two experiments match `assignment.py`. Our five classes match the paper, but our saved task-family split differs. AI assistance was used to prepare the code and writing. Dataset research-use permission still requires confirmation.

[Our repository](https://github.com/S3eeDTR/LLM-AuthorBench-Assignment) | [Dataset source](https://github.com/LLMauthorbench/LLMauthorbench)


## Setup
Use Colab's preinstalled scientific libraries instead of replacing NumPy or pandas. This keeps Colab's own dependencies compatible. The versions below are recorded with your results; they may differ from the local benchmark environment.

**If you already ran the older installation cell:** use **Runtime → Disconnect and delete runtime**, reconnect, then run this updated notebook. A simple restart does not undo installed packages. Save any needed runtime files first; Drive files persist.


In [ ]:
import importlib.metadata

# Check actual imports, not just the installed version labels.
try:
    import numpy as np
    import pandas as pd
    from sklearn.feature_extraction.text import TfidfVectorizer
    TfidfVectorizer().fit_transform(["check numpy imports", "check sklearn imports"])
except (ImportError, ValueError, AttributeError) as error:
    raise RuntimeError(
        "The runtime has incompatible or mixed scientific packages. "
        "Save needed files, choose Runtime > Disconnect and delete runtime, "
        "then reopen the latest GitHub notebook and reconnect. "
        "Do not run the old pip installation cell."
    ) from error

package_versions = {
    name: importlib.metadata.version(name)
    for name in ("numpy", "pandas", "scipy", "scikit-learn", "joblib")
}
for name, version in package_versions.items():
    print(f"{name}: {version}")
print("Scientific library import check passed.")


## Shared code
Imports, the comment-removal function and one training function. Using that same function twice keeps both experiments comparable.

In [ ]:
"""Run the four-model assignment and its comment-removal experiment."""
from pathlib import Path
import hashlib
import json
import os
import re
import zipfile
import urllib.request
from time import perf_counter

ROOT = Path.cwd() / "LLM-AuthorBench-Assignment"
ROOT.mkdir(exist_ok=True)
OUTPUT = ROOT / "results"
OUTPUT.mkdir(exist_ok=True)
(OUTPUT / "colab_environment.json").write_text(json.dumps(package_versions, indent=2))
# Keep temporary files beside the project and limit CPU thread usage.
(ROOT / "tmp").mkdir(exist_ok=True)
for name in ("TEMP", "TMP", "TMPDIR", "JOBLIB_TEMP_FOLDER"):
    os.environ[name] = str(ROOT / "tmp")
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[name] = "4"

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42

# Match strings and character literals as well as comments, so quoted // stays intact.
TOKEN = re.compile(r'"(?:\\.|[^"\\])*"|\'(?:\\.|[^\'\\])*\'|//[^\n]*(?:\n|$)|/\*.*?\*/', re.S)


def remove_comments(code):
    code = code.replace('\\\r\n', '').replace('\\\n', '')

    def replace(match):
        text = match.group()
        if text.startswith('//') or text.startswith('/*'):
            return ' ' + '\n' * text.count('\n')
        return text

    return TOKEN.sub(replace, code)


def run_experiment(data, condition):
    """Use the same steps for both experiments so the comparison is fair."""
    labels = sorted(data.label.unique())
    results = []
    folder_name = "experiment_1_original" if condition == "original" else "experiment_2_no_comments"
    folder = OUTPUT / folder_name
    folder.mkdir(exist_ok=True)
    model_folder = ROOT / "models" / folder_name
    model_folder.mkdir(parents=True, exist_ok=True)
    code = data.code if condition == "original" else data.code.map(remove_comments)
    vectorizer = TfidfVectorizer(
        analyzer="char", ngram_range=(3, 5), max_features=50000,
        sublinear_tf=True, lowercase=False, dtype=np.float32,
    )
    started = perf_counter()
    x = {"train": vectorizer.fit_transform(code[data.split == "train"])}
    feature_seconds = perf_counter() - started
    for part in ("validation", "test"):
        x[part] = vectorizer.transform(code[data.split == part])
    joblib.dump(vectorizer, model_folder / "vectorizer.joblib", compress=3)
    (folder / "features.json").write_text(json.dumps({
        "training_feature_seconds": feature_seconds,
        "features": len(vectorizer.vocabulary_),
        "vectorizer_MB": (model_folder / "vectorizer.joblib").stat().st_size / 1e6,
    }, indent=2))

    # Train the four models on exactly the same features.
    models = {
        "Naive Bayes": MultinomialNB(alpha=1.0),
        "Logistic Regression": LogisticRegression(C=1.0, max_iter=3000, solver="lbfgs", random_state=SEED),
        "Linear SVM": LinearSVC(C=1.0, dual="auto", max_iter=10000, random_state=SEED),
        "Random Forest": RandomForestClassifier(
            n_estimators=150, max_depth=40, min_samples_leaf=2,
            max_features="sqrt", n_jobs=4, random_state=SEED,
        ),
    }
    for name, model in models.items():
        print(f"{condition}: training {name}", flush=True)
        started = perf_counter()
        model.fit(x["train"], data.loc[data.split == "train", "label"])
        training_seconds = perf_counter() - started
        model_path = model_folder / f"{name}.joblib"
        joblib.dump(model, model_path, compress=3)

        # Measure each model and save its predictions.
        for part in ("validation", "test"):
            rows = data[data.split == part]
            started = perf_counter()
            predicted = model.predict(x[part])
            inference_seconds = perf_counter() - started
            precision, recall, f1, _ = precision_recall_fscore_support(
                rows.label, predicted, average="macro", zero_division=0,
            )
            results.append({
                "condition": condition, "model": name, "split": part,
                "accuracy": accuracy_score(rows.label, predicted),
                "macro_f1": f1, "macro_precision": precision, "macro_recall": recall,
                "training_seconds": training_seconds, "inference_seconds": inference_seconds,
                "model_MB": model_path.stat().st_size / 1e6,
            })
            pd.DataFrame({"row_id": rows.row_id, "true": rows.label, "predicted": predicted}).to_csv(
                folder / f"{name}_{part}_predictions.csv", index=False,
            )
            pd.DataFrame(classification_report(rows.label, predicted, output_dict=True, zero_division=0)).T.to_csv(
                folder / f"{name}_{part}_classes.csv",
            )
            pd.DataFrame(confusion_matrix(rows.label, predicted, labels=labels), index=labels, columns=labels).to_csv(
                folder / f"{name}_{part}_confusion.csv",
            )
    return results




## Data preparation
Download the dataset and the saved split. Check the five labels, duplicates and separation between splits. Inferred task groups are conservative estimates, not official task identifiers.

In [ ]:
split_file = ROOT / "data/splits.csv"
split_file.parent.mkdir(exist_ok=True)
if not split_file.exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/S3eeDTR/LLM-AuthorBench-Assignment/550295d240b5aadb023d6656c3561fe09c9f6041/data/splits.csv", split_file)

# DATA: load the five classes and check the saved split.
archive = ROOT / "data/LLM-AuthorBench.json.zip"
if not archive.exists():
    archive.parent.mkdir(exist_ok=True)
    url = "https://raw.githubusercontent.com/LLMauthorbench/LLMauthorbench/6a1c2ac173c774cfbd012f4c81e8d0a51bb61eb7/LLM-AuthorBench.json.zip"
    urllib.request.urlretrieve(url, archive)
assert hashlib.sha256(archive.read_bytes()).hexdigest() == "e24399b7b05b812c5a1148ef44245348859eaa74b1140523cb695f984337f24c"
with zipfile.ZipFile(archive) as file:
    records = json.loads(file.read("LLM-AuthorBench.json"))
data = pd.DataFrame({
    "row_id": range(len(records)),
    "code": [row["c_code"] for row in records],
    "label": [row["model_name"] for row in records],
    "prompt": [row["prompt"] for row in records],
})
data["source_hash"] = data.code.map(lambda code: hashlib.sha256(code.encode()).hexdigest())
data = data[data.label.isin(["gpt-4.1", "deepseek-chat", "claude-3.5-haiku",
                                "gemini-2.5-flash-preview-05-20", "llama-3.3-70b-instruct"])]
assert data.code.str.strip().ne("").all()
assert not data.source_hash.duplicated().any()
split = pd.read_csv(ROOT / "data/splits.csv")
data = data.merge(split, on=["row_id", "label", "source_hash"], validate="one_to_one")
assert len(data) == len(split) == 20000
labels = sorted(data.label.unique())
assert len(labels) == 5
assert set(data.split) == {"train", "validation", "test"}
for part in ("train", "validation", "test"):
    assert set(data.loc[data.split == part, "label"]) == set(labels)
    for column in ("problem_id", "prompt", "source_hash"):
        assert not set(data.loc[data.split == part, column]) & set(data.loc[data.split != part, column])
print(data.groupby("split").size(), flush=True)
pd.crosstab(data.split, data.label).to_csv(OUTPUT / "class_counts.csv")
data.groupby("split").agg(samples=("row_id", "size"), groups=("problem_id", "nunique")).to_csv(OUTPUT / "split_counts.csv")



## Experiment 1: Original code
Train Naive Bayes, Logistic Regression, Linear SVM and Random Forest using one training-only TF-IDF representation. Save validation and test scores.

In [ ]:
# EXPERIMENT 1: Original source code, including comments.
print("\nEXPERIMENT 1: ORIGINAL CODE", flush=True)
results = run_experiment(data, "original")



## Experiment 2: Code without comments
Repeat with comments removed. Keep split, seed and model settings fixed.

Exploratory prediction: removing comments will narrow the SVM validation Macro-F1 lead over Logistic Regression and may reverse their ranking. Earlier eight-class results were already known; this is not an independent preregistration. Report the measured outcome whether or not the prediction succeeds.

In [ ]:
# EXPERIMENT 2: Remove comments and repeat with the same settings.
print("\nEXPERIMENT 2: CODE WITHOUT COMMENTS", flush=True)
results += run_experiment(data, "comments_removed")



## Final comparison
Select the winner using original validation Macro-F1, then report test scores. Compare the score changes and ranks across the two experiments.

In [ ]:
# FINAL COMPARISON: select with validation scores and report changes.
results = pd.DataFrame(results)
results.to_csv(OUTPUT / "metrics.csv", index=False)
validation = results[(results.condition == "original") & (results.split == "validation")]
winner = validation.sort_values(["macro_f1", "model"], ascending=[False, True]).iloc[0].model
(OUTPUT / "winner.txt").write_text(f"Highest original validation Macro-F1: {winner}\n")
comparison = results.pivot(index=["model", "split"], columns="condition", values="macro_f1")
comparison["change_percentage_points"] = 100 * (comparison.comments_removed - comparison.original)
comparison["original_rank"] = comparison.groupby(level="split").original.rank(ascending=False)
comparison["comments_removed_rank"] = comparison.groupby(level="split").comments_removed.rank(ascending=False)
comparison.to_csv(OUTPUT / "comment_comparison.csv")
print("Validation-selected winner:", winner)
print(results[results.split == "test"][["condition", "model", "accuracy", "macro_f1"]].to_string(index=False))
print("Results saved in", OUTPUT)


## Experiment 3: CodeT5-Authorship (optional GPU experiment)

Run **Setup, Shared code and Data preparation** first. You may skip Experiments 1 and 2 if you only want CodeT5: the final comparison can load their recorded results.

Choose **Runtime → Change runtime type → GPU before starting**. Changing runtime can clear variables. This trains the authors' CodeT5+ encoder with a first-token classification head on our **same five classes and split**, using original code with comments. It is not an exact reproduction of their random-split experiment.

Ten epochs can still take many hours. This is one neural run, without a second comment-removal run. Drive storage is enabled to retain completed-epoch checkpoints. Colab will ask for Drive access. Setting it to False uses temporary runtime storage instead. Download the results at the end.

Sources: [authors' implementation](https://github.com/LLMauthorbench/LLMauthorbench/blob/6a1c2ac173c774cfbd012f4c81e8d0a51bb61eb7/scripts/5_CodeT5-Authorship_5-class_google_colab.ipynb), [base model](https://huggingface.co/Salesforce/codet5p-770m), [Trainer documentation](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer).

### 3A. Install and configure
The model is the 770M-named encoder-decoder checkpoint with only its encoder retained, not a smaller replacement. Default length 512 follows the paper table; the released code uses 1024. Microbatch 1 × accumulation 32 preserves effective batch size 32. Change the run name if settings change. No neural score has been generated yet.

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime, then rerun setup and data preparation."
import sys, subprocess, importlib.metadata
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "transformers==4.57.1", "accelerate==1.10.1", "sentencepiece==0.2.1"])
from packaging.version import Version
assert Version(torch.__version__.split("+")[0]) >= Version("2.6"), "Use a current Colab PyTorch runtime."
from torch import nn
from transformers import AutoTokenizer, T5EncoderModel, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding, set_seed
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers.trainer_utils import get_last_checkpoint

EPOCHS = 10
MAX_LENGTH = 512
USE_DRIVE = True
RUN_NAME = "codet5-five-classes-512-seed42"
MODEL_NAME = "Salesforce/codet5p-770m"
MODEL_VERSION = "3f7cc6e80aee6612f41253b5ee28f09f86faa35f"
set_seed(SEED)

save_root = ROOT / "models"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    save_root = Path("/content/drive/MyDrive/LLM-AuthorBench")
run_folder = save_root / RUN_NAME
run_folder.mkdir(parents=True, exist_ok=True)
neural_output = OUTPUT / "experiment_3_codet5"
neural_output.mkdir(exist_ok=True)
settings = {"model": MODEL_NAME, "revision": MODEL_VERSION, "epochs": EPOCHS,
            "max_length": MAX_LENGTH, "seed": SEED, "batch": 1, "accumulation": 32,
            "learning_rate": 5e-5, "bf16": torch.cuda.is_bf16_supported(),
            "split_hash": hashlib.sha256(data[["row_id", "split", "label"]].to_csv(index=False).encode()).hexdigest()}
config_path = run_folder / "settings.json"
if config_path.exists():
    assert json.loads(config_path.read_text()) == settings, "Settings changed: choose a new RUN_NAME."
config_path.write_text(json.dumps(settings, indent=2))
(neural_output / "settings.json").write_text(json.dumps(settings, indent=2))
print("GPU:", torch.cuda.get_device_name(0), "Checkpoint folder:", run_folder)

### 3B. Tokenize and define the model
Only source code enters the tokenizer, prefixed with `code: ` as in the authors' notebook. Inputs longer than the token limit are truncated. All encoder layers are fine-tuned. The head uses a 768-unit layer, GELU and dropout 0.2, followed by five output classes.

In [ ]:
labels = sorted(data.label.unique())
label_ids = {label: i for i, label in enumerate(labels)}
rows = {part: data[data.split == part].reset_index(drop=True)
        for part in ("train", "validation", "test")}
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_VERSION)

class CodeDataset(torch.utils.data.Dataset):
    def __init__(self, frame):
        self.tokens = tokenizer(["code: " + code for code in frame.code],
                                truncation=True, max_length=MAX_LENGTH)
        self.targets = frame.label.map(label_ids).tolist()
    def __len__(self):
        return len(self.targets)
    def __getitem__(self, index):
        item = {name: values[index] for name, values in self.tokens.items()}
        item["labels"] = self.targets[index]
        return item

encoded = {part: CodeDataset(frame) for part, frame in rows.items()}

class CodeT5Authorship(nn.Module):
    accepts_loss_kwargs = False
    def __init__(self):
        super().__init__()
        self.encoder = T5EncoderModel.from_pretrained(MODEL_NAME, revision=MODEL_VERSION, dtype=torch.float32)
        self.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        self.encoder.config.use_cache = False
        self.head = nn.Sequential(nn.Linear(self.encoder.config.d_model, 768),
                                  nn.GELU(), nn.Dropout(0.2), nn.Linear(768, len(labels)))
    def forward(self, input_ids, attention_mask=None, labels=None):
        first_token = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        logits = self.head(first_token)
        loss = nn.functional.cross_entropy(logits.float(), labels) if labels is not None else None
        return SequenceClassifierOutput(loss=loss, logits=logits)

def neural_metrics(prediction):
    predicted = prediction.predictions.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        prediction.label_ids, predicted, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(prediction.label_ids, predicted),
            "macro_f1": f1, "macro_precision": precision, "macro_recall": recall}

model = CodeT5Authorship()

### 3C. Train
The best epoch is selected using validation Macro-F1. Test data is not used for epoch selection. Checkpoints are saved after each completed epoch; rerunning resumes the last saved checkpoint. An interrupted part of an epoch must be repeated. BF16 mixed precision is used when supported; other GPUs use FP32, which can be slower. If memory runs out, restart, reduce the token limit with a new run name, and document the change.

In [ ]:
arguments = TrainingArguments(
    output_dir=str(run_folder / "checkpoints"), num_train_epochs=EPOCHS,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=32, learning_rate=5e-5,
    weight_decay=0.01, warmup_steps=200, lr_scheduler_type="cosine_with_restarts",
    optim="adamw_torch", eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    greater_is_better=True, save_total_limit=2, save_safetensors=False,
    bf16=settings["bf16"], fp16=False, report_to="none",
    seed=SEED, data_seed=SEED, logging_steps=25, logging_nan_inf_filter=False,
)
trainer = Trainer(model=model, args=arguments,
                  train_dataset=encoded["train"], eval_dataset=encoded["validation"],
                  data_collator=DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8),
                  compute_metrics=neural_metrics)
checkpoint_dir = run_folder / "checkpoints"
last_checkpoint = get_last_checkpoint(str(checkpoint_dir)) if checkpoint_dir.exists() else None
torch.cuda.synchronize()
started = perf_counter()
trainer.train(resume_from_checkpoint=last_checkpoint)
torch.cuda.synchronize()
training_seconds = perf_counter() - started
history = pd.DataFrame(trainer.state.log_history)
assert np.isfinite(history["loss"].dropna()).all(), "Non-finite loss: do not report this run."
history.to_csv(neural_output / "training_history.csv", index=False)
trainer.save_model(str(run_folder / "best_model"))
tokenizer.save_pretrained(run_folder / "best_model")
(run_folder / "best_model" / "labels.json").write_text(json.dumps(labels))

### 3D. Evaluate and compare
Save actual scores for validation and test, then compare against Experiment 1. The comparison measures whole systems: CodeT5 uses pretraining and truncated tokens, while the classical models use full-source character TF-IDF. GPU time and neural model size are not directly comparable to CPU time and compressed classical sizes. A win is not guaranteed. There is no neural comment-removal run in this notebook.

In [ ]:
neural_rows = []
model_mb = sum(p.stat().st_size for p in (run_folder / "best_model").rglob("*") if p.is_file()) / 1e6
for part in ("validation", "test"):
    prediction = trainer.predict(encoded[part])
    predicted_ids = prediction.predictions.argmax(axis=-1)
    neural_rows.append({"condition": "original", "model": "CodeT5-Authorship", "split": part,
                       **neural_metrics(prediction), "training_seconds": training_seconds if last_checkpoint is None else None,
                       "session_training_seconds": training_seconds, "model_MB": model_mb})
    pd.DataFrame({"row_id": rows[part].row_id, "true": rows[part].label,
                  "predicted": [labels[i] for i in predicted_ids]}).to_csv(
                      neural_output / f"{part}_predictions.csv", index=False)
neural_results = pd.DataFrame(neural_rows)
neural_results.to_csv(neural_output / "metrics.csv", index=False)
(neural_output / "environment.json").write_text(json.dumps({
    "gpu": torch.cuda.get_device_name(0), "resumed": last_checkpoint is not None,
    "best_validation_macro_f1": trainer.state.best_metric,
    "packages": {name: importlib.metadata.version(name) for name in ("torch", "transformers", "accelerate")},
    "timing_note": "Includes epoch validation and checkpoint saving; interrupted total time is unknown."
}, indent=2))
# Keep a small copy in Drive alongside the checkpoints when Drive is enabled.
import shutil
shutil.copytree(neural_output, run_folder / "results", dirs_exist_ok=True)

baseline_path = OUTPUT / "metrics.csv"
if baseline_path.exists():
    baselines = pd.read_csv(baseline_path)
else:
    baselines = pd.read_csv("https://raw.githubusercontent.com/S3eeDTR/LLM-AuthorBench-Assignment/550295d240b5aadb023d6656c3561fe09c9f6041/results/metrics.csv")
comparison = pd.concat([baselines[baselines.condition == "original"], neural_results], ignore_index=True)
comparison.to_csv(OUTPUT / "five_model_comparison.csv", index=False)
display(comparison[comparison.split == "test"][["model", "accuracy", "macro_f1", "training_seconds", "model_MB"]])
validation = comparison[comparison.split == "validation"]
print("Validation-selected winner:", validation.sort_values(["macro_f1", "model"], ascending=[False, True]).iloc[0].model)
print("Interpret only the measured results. This does not reproduce the paper's split or isolate pretraining's effect.")

## Download results
The ZIP contains metrics, predictions and confusion matrices. Trained models stay in the runtime. Download before closing Colab.

In [ ]:
import shutil
archive = shutil.make_archive(str(ROOT / "results"), "zip", OUTPUT)
from google.colab import files
files.download(archive)
